# Pipeline de Processamento e Limpeza de Dados

Este notebook apresenta a análise, limpeza, normalização e integração das bases de dados para o projeto de acidentes de trânsito em rodovias federais. Ele ilustra os processamentos necessários para preparar os dados antes de subi-los ao banco.

## Conjuntos de dados abordados:
1. **Acidentes** (`datatran2026.csv`)
2. **Fiscalização (PNCV Radares)** (`pncv_dados_abertos_12_07_2024.xlsx` e históricos)
3. **Sinalização (Placas de Proibido Ultrapassar)** (`proibido-ultrapassar-16-06-2026.json`)


In [1]:
import os
import pandas as pd
import numpy as np
import json

# Definir caminhos relativos para as bases de dados
DADOS_DIR = os.path.join("..", "..", "dados")
print("Diretório de dados detectado:", os.path.abspath(DADOS_DIR))


Diretório de dados detectado: /home/henriuz/Documentos/01-unifei/07-periodo/spad03-introducao-a-analise-de-dados/projeto-acidentes-transito/dados


## 1. Dados de Acidentes (datatran2026.csv)

### Problemas Identificados:
- **Codificação:** O arquivo usa `latin1` em vez de `utf-8`.
- **Tipagem Numérica:** Decimais de coordenadas (`latitude` e `longitude`) e quilometragem (`km`) usam vírgula (ex: `-7,291548`).
- **Chave de Integração:** O identificador da rodovia `br` é lido como inteiro (ex: `153`) e precisa ser padronizado para string de 3 dígitos com zeros à esquerda (ex: `"153"` ou `"060"`) para integrar com os demais arquivos.
- **Valores Nulos:** alguns acidentes possuem colunas com valores desconhecidos (`NaN`). O correto é deixar claro que o valor é desconhecido, porém no caso da coluna "regional", o tratamento deve ser o valor "SPRF-" + o valor da coluna "uf".


In [10]:
acidentes_path = os.path.join(DADOS_DIR, "acidentes", "datatran2026.csv")

print("=== DADOS DE ACIDENTES (PRÉ-PROCESSAMENTO) ===")
df_acid_raw = pd.read_csv(acidentes_path, sep=';', encoding='latin1')
print(f"Número de colunas com valores nulos: {(df_acid_raw.isna().sum() > 0).sum()}")
print(df_acid_raw.head(5).to_string())

# Executando o processamento
df_acid = pd.read_csv(acidentes_path, sep=';', encoding='latin1')

# 1. Substituir vírgula por ponto e converter para numérico
df_acid['latitude'] = pd.to_numeric(df_acid['latitude'].astype(str).str.replace(',', '.'), errors='coerce')
df_acid['longitude'] = pd.to_numeric(df_acid['longitude'].astype(str).str.replace(',', '.'), errors='coerce')
df_acid['km'] = pd.to_numeric(df_acid['km'].astype(str).str.replace(',', '.'), errors='coerce')

# 2. Padronizar a rodovia BR para string de 3 dígitos (ex: 60 -> '060', 153 -> '153')
df_acid['br'] = df_acid['br'].astype(str).str.replace('.0', '', regex=False).str.strip().str.zfill(3)

# 3. Substituir NaN por "Desconhecido" nas colunas correspondentes
colunas_nulas = ['classificacao_acidente', 'delegacia', 'uop']
df_acid[colunas_nulas] = df_acid[colunas_nulas].fillna('Desconhecido')

# 4. Substituir NaN por SPRF- mais o valor na coluna uf
mascara = df_acid['regional'].isna()
df_acid.loc[mascara, 'regional'] = 'SPRF-' + df_acid.loc[mascara, 'uf']

print("\n=== DADOS DE ACIDENTES (PÓS-PROCESSAMENTO) ===")
print(f"Número de colunas com valores nulos: {(df_acid.isna().sum() > 0).sum()}")
print(df_acid.head(5).to_string())


=== DADOS DE ACIDENTES (PRÉ-PROCESSAMENTO) ===
Número de colunas com valores nulos: 4
       id data_inversa    dia_semana   horario  uf   br     km  municipio                            causa_acidente                  tipo_acidente classificacao_acidente   fase_dia  sentido_via condicao_metereologica tipo_pista  tracado_via uso_solo  pessoas  mortos  feridos_leves  feridos_graves  ilesos  ignorados  feridos  veiculos      latitude     longitude regional delegacia             uop
0  742921   2026-01-01  quinta-feira  04:04:00  TO  153    155  ARAGUAINA  Objeto estático sobre o leito carroçável                     Tombamento                    NaN  Amanhecer    Crescente              Céu Claro    Simples  Aclive;Reta      Não        3       1              0               0       1          3        0         5     -7,291548    -48,286252  SPRF-TO  DEL02-TO  UOP01-DEL02-TO
1  742942   2026-01-01  quinta-feira  06:40:00  MG  262  146,1  RIO CASCA                         Condutor Dormindo 

## 2. Radares de Fiscalização (PNCV 2024) - Extração das Fórmulas e Tratamento de Falhas

### Solução do Bug Crítico de Coordenadas:
No arquivo Excel `pncv_dados_abertos_12_07_2024.xlsx`, as coordenadas foram salvas com o caractere `/` como divisor (ex: `=-10.011417 / -67.794678`), fazendo com que o Excel interpretasse como uma divisão matemática.

Ao ler as fórmulas cruas com `engine_kwargs={'data_only': False}`, conseguimos obter as strings originais. Contudo, há **8 radares** (no AP e RR) que não iniciam com `=` e usam vírgula como decimal (ex: `'0,082013 / -51,076582'`). Um parser básico que apenas tenta converter para float após dividir a string irá falhar nessas linhas.

Abaixo, demonstramos o fluxo de processamento em duas etapas:
1. A visão inicial do DataFrame carregada normalmente (coordenadas divididas e corrompidas).
2. A visão do DataFrame com as strings de fórmulas carregadas.
3. O pós-processamento inicial (extração básica das fórmulas normais que resulta em 8 falhas).
4. A listagem das tuplas que falharam devido à vírgula decimal.
5. O processamento corretivo adicional focado em limpar e resolver esses 8 casos com vírgula para atingir 100% de recuperação.


In [1]:
p2024_path = os.path.join(DADOS_DIR, "fiscalizacao", "pncv_dados_abertos_12_07_2024.xlsx")

# 1. PRÉ-PROCESSAMENTO - Padrão Pandas (valores calculados corrompidos)
print("=== RADARES PNCV 2024 (PRÉ-PROCESSAMENTO - PADRÃO PANDAS / VALOR CALCULADO CORROMPIDO) ===")
df_24_calc = pd.read_excel(p2024_path, nrows=5)
df_24_calc.columns = [c.strip() for c in df_24_calc.columns]
print(df_24_calc[['Código Identificação', 'UF', 'Rodovia', 'Km', 'Coordenadas (Lat/Long)']].to_string())

# 2. PRÉ-PROCESSAMENTO - Fórmulas extraídas do Excel
print("\n=== RADARES PNCV 2024 (PRÉ-PROCESSAMENTO - COM FÓRMULAS EXTRAÍDAS) ===")
df_24 = pd.read_excel(p2024_path, engine_kwargs={'data_only': False})
df_24.columns = [c.strip() for c in df_24.columns]
print(df_24[['Código Identificação', 'UF', 'Rodovia', 'Km', 'Coordenadas (Lat/Long)']].head(5).to_string())

# 3. PÓS-PROCESSAMENTO INICIAL - Extração básica (sem tratar as vírgulas decimais)
def extract_coords_basic(val):
    if pd.isnull(val): return None, None
    val_str = str(val).strip()
    if val_str.startswith('='):
        val_str = val_str[1:]
    parts = val_str.split('/')
    if len(parts) == 2:
        try:
            lat = float(parts[0].strip())
            lon = float(parts[1].strip())
            return lat, lon
        except ValueError:
            pass
    return None, None

lats_basic, lons_basic = [], []
for val in df_24['Coordenadas (Lat/Long)']:
    lat, lon = extract_coords_basic(val)
    lats_basic.append(lat)
    lons_basic.append(lon)

df_24['latitude'] = lats_basic
df_24['longitude'] = lons_basic
df_24['km'] = pd.to_numeric(df_24['Km'].astype(str).str.replace(',', '.'), errors='coerce')
df_24['br'] = df_24['Rodovia'].astype(str).str.replace('BR-', '', regex=False).str.strip().str.zfill(3)

print("\n=== RADARES PNCV 2024 (PÓS-PROCESSAMENTO INICIAL - SEM TRATAR VÍRGULA) ===")
print(df_24[['Código Identificação', 'UF', 'br', 'km', 'latitude', 'longitude']].head(5).to_string())
print(f"Recuperados com sucesso no passo inicial: {df_24['latitude'].notnull().sum()} de {len(df_24)} radares.")

# 4. TUPLAS COM ERRO DE CONVERSÃO (Linhas que falharam no passo inicial devido à vírgula)
df_failed = df_24[df_24['latitude'].isnull()]
print("\n=== TUPLAS COM ERRO DE CONVERSÃO (RADAR E VALOR DA CÉLULA) ===")
print(f"Total de falhas devido à vírgula: {len(df_failed)}")
for idx, row in df_failed.iterrows():
    print(f"  Código Identificação: {row['Código Identificação']} | Coordenadas: {repr(row['Coordenadas (Lat/Long)'])} ")

# 5. PÓS-PROCESSAMENTO DE CORREÇÃO - Tratamento específico para os casos de vírgula
print("\n=== PÓS-PROCESSAMENTO DE CORREÇÃO (APLICANDO O TRATAMENTO DE VÍRGULA PARA OS ERROS) ===")
def extract_coords_robust(val):
    if pd.isnull(val): return None, None
    val_str = str(val).strip()
    if val_str.startswith('='):
        val_str = val_str[1:]
    parts = val_str.split('/')
    if len(parts) == 2:
        try:
            # Substitui a vírgula por ponto para permitir conversão para float
            lat = float(parts[0].replace(',', '.').strip())
            lon = float(parts[1].replace(',', '.').strip())
            return lat, lon
        except ValueError:
            pass
    return None, None

# Atualiza apenas as linhas que falharam no passo básico anterior
recovered_lats = df_24['latitude'].copy()
recovered_lons = df_24['longitude'].copy()

for idx in df_failed.index:
    val = df_24.loc[idx, 'Coordenadas (Lat/Long)']
    lat, lon = extract_coords_robust(val)
    recovered_lats.loc[idx] = lat
    recovered_lons.loc[idx] = lon

df_24['latitude'] = recovered_lats
df_24['longitude'] = recovered_lons

print("Radares corrigidos e recuperados com sucesso:")
print(df_24.loc[df_failed.index, ['Código Identificação', 'UF', 'br', 'km', 'latitude', 'longitude']].to_string())

total_rows = len(df_24)
rec_count = df_24['latitude'].notnull().sum()
print(f"\nRecuperação final após correção: {rec_count} de {total_rows} radares com coordenadas restauradas ({rec_count/total_rows*100:.2f}%).")


=== RADARES PNCV 2024 (PRÉ-PROCESSAMENTO - PADRÃO PANDAS / VALOR CALCULADO CORROMPIDO) ===
  Código Identificação  UF  Rodovia       Km  Coordenadas (Lat/Long)
0          ACB22010002  AC      364  124.400                0.147673
1          ALB18080001  AL      104   33.740                0.254106
2          ALB18080002  AL      104   86.255                0.264228
3          ALB18080003  AL      104   87.257                0.264484
4          ALB18080004  AL      104   87.257                0.264484

=== RADARES PNCV 2024 (PRÉ-PROCESSAMENTO - COM FÓRMULAS EXTRAÍDAS) ===
  Código Identificação  UF  Rodovia       Km           Coordenadas (Lat/Long)
0          ACB22010002  AC      364  124.400         =-10.011417 / -67.794678
1          ALB18080001  AL      104   33.740  =-9.15115199999999 / -36.013171
2          ALB18080002  AL      104   86.255          =-9.466008 / -35.825153
3          ALB18080003  AL      104   87.257   =-9.474353 / -35.8220549999999
4          ALB18080004  AL      1

## 3. Dados de Sinalização - Placas de Proibido Ultrapassar (proibido-ultrapassar-16-06-2026.json)

### Problemas Identificados:
- **Formato:** O arquivo está estruturado em JSON com a lista de placas na chave `"proibidoUltrapassar"`.
- **Tipagem:** Coordenadas de início/fim e quilometragens usam a vírgula para decimais e são strings.
- **Normalização:** A rodovia está nomeada como `"BR-101"`, necessitando de extração numérica para padronizar como `"101"`.


In [1]:
json_path = os.path.join(DADOS_DIR, "fiscalizacao", "proibido-ultrapassar-16-06-2026.json")

print("=== SINALIZAÇÃO PLACAS (PRÉ-PROCESSAMENTO) ===")
with open(json_path, 'r', encoding='utf-8') as f:
    json_data = json.load(f)
df_json_raw = pd.DataFrame(json_data['proibidoUltrapassar'][:5])
print(df_json_raw[['concessionaria', 'uf', 'rodovia', 'km_m_inicio', 'latitude_inicio', 'longitude_inicio', 'km_m_final']].to_string())

# Processamento
df_json = pd.DataFrame(json_data['proibidoUltrapassar'])

# 1. Conversão dos decimais em string para float
for col in ['km_m_inicio', 'latitude_inicio', 'longitude_inicio', 'km_m_final', 'latitude_final', 'longitude_final']:
    df_json[col] = pd.to_numeric(df_json[col].astype(str).str.replace(',', '.'), errors='coerce')

# 2. Padronizar a rodovia
df_json['br'] = df_json['rodovia'].astype(str).str.replace('BR-', '', regex=False).str.strip().str.zfill(3)

print("\n=== SINALIZAÇÃO PLACAS (PÓS-PROCESSAMENTO) ===")
print(df_json[['concessionaria', 'uf', 'br', 'km_m_inicio', 'latitude_inicio', 'longitude_inicio', 'km_m_final']].head(5).to_string())


=== SINALIZAÇÃO PLACAS (PRÉ-PROCESSAMENTO) ===
         concessionaria  uf rodovia km_m_inicio latitude_inicio longitude_inicio km_m_final
0  AUTOPISTA FLUMINENSE  RJ  BR-101       0,000      -21,222617       -41,309502      2,140
1  AUTOPISTA FLUMINENSE  RJ  BR-101       2,390      -21,238400       -41,322779      2,690
2  AUTOPISTA FLUMINENSE  RJ  BR-101       3,160      -21,244189       -41,326372      4,310
3  AUTOPISTA FLUMINENSE  RJ  BR-101       4,460      -21,255628       -41,324991      4,940
4  AUTOPISTA FLUMINENSE  RJ  BR-101       5,180      -21,261610       -41,326905      5,740

=== SINALIZAÇÃO PLACAS (PÓS-PROCESSAMENTO) ===
         concessionaria  uf   br  km_m_inicio  latitude_inicio  longitude_inicio  km_m_final
0  AUTOPISTA FLUMINENSE  RJ  101         0.00       -21.222617        -41.309502        2.14
1  AUTOPISTA FLUMINENSE  RJ  101         2.39       -21.238400        -41.322779        2.69
2  AUTOPISTA FLUMINENSE  RJ  101         3.16       -21.244189        -41.

## 4. Conclusão e Prontidão das Chaves de Integração

Ao utilizar a opção `engine_kwargs={'data_only': False}` no Pandas e tratar as vírgulas decimais das strings, fomos capazes de acessar a fórmula original do Excel e extrair as coordenadas reais de todos os 1.980 radares do PNCV de 2024 de forma nativa e simples!

Agora, todos os três conjuntos de dados (Acidentes, Radares e Placas de Sinalização) possuem dados numéricos de coordenadas (`latitude` e `longitude`), quilometragens (`km`) e rodovias padronizadas (`br` com 3 dígitos) prontos para cruzamentos espaciais e persistência no banco relacional!
